# Scraper Progress Monitor

Live-Ansicht des aktuell laufenden (oder zuletzt gelaufenen) Willhaben-Scrapers. Die Fortschrittsdaten werden vom Scraper pro Seite in die Collection `scraper_runs` geschrieben.

In [ ]:
import os
import time
from datetime import datetime, timezone

import pandas as pd
from IPython.display import HTML, clear_output, display
from pymongo import MongoClient

host = os.environ.get("MONGO_HOST", "localhost")
port = int(os.environ.get("MONGO_PORT", 27017))
user = os.environ.get("MONGO_USER", "admin")
password = os.environ.get("MONGO_PASSWORD", "BigDataMongo2026!")

client = MongoClient(host=host, port=port, username=user, password=password, authSource="admin")
db = client["bigdata"]
runs = db["scraper_runs"]

print(f"{runs.count_documents({})} Runs in der DB")

In [ ]:
def format_duration(seconds):
    seconds = max(0, int(seconds))
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    if h:
        return f"{h}h {m}m {s}s"
    if m:
        return f"{m}m {s}s"
    return f"{s}s"


def render_run(run):
    if not run:
        return HTML("<b>Noch kein Run in der DB vorhanden.</b>")

    status = run.get("status", "?")
    page = run.get("current_page", 0) or 0
    max_pages = run.get("max_pages", 1) or 1
    pages_done = run.get("pages_completed", 0) or 0
    pct = min(100.0, (page / max_pages) * 100) if max_pages else 0

    started = run.get("started_at")
    last_update = run.get("last_update") or started
    finished = run.get("finished_at")

    now = datetime.now(timezone.utc)
    elapsed_s = (finished - started).total_seconds() if finished and started else (now - started).total_seconds() if started else 0

    eta_str = "\u2014"
    if status == "running" and pages_done > 0 and elapsed_s > 0:
        per_page = elapsed_s / pages_done
        remaining = max(0, max_pages - pages_done)
        eta_str = format_duration(per_page * remaining)

    status_color = {"running": "#1565C0", "completed": "#2E7D32", "failed": "#B71C1C"}.get(status, "#333")

    bar_width = 420
    filled = int(bar_width * pct / 100)

    html = f"""
    <div style="font-family: sans-serif; max-width:640px;">
      <h3 style="margin-bottom:6px;">Scraper-Run
        <span style="color:{status_color}; font-size:0.85em;">[{status.upper()}]</span>
      </h3>
      <div style="background:#eee; border-radius:4px; width:{bar_width}px; height:24px; margin:8px 0;">
        <div style="background:{status_color}; width:{filled}px; height:24px; border-radius:4px;
                    text-align:center; color:white; line-height:24px; font-size:12px;">
          {pct:.1f}%
        </div>
      </div>
      <table style="font-size:13px; border-collapse:collapse;">
        <tr><td style="padding:2px 12px 2px 0;"><b>Seite:</b></td><td>{page} / {max_pages}</td></tr>
        <tr><td style="padding:2px 12px 2px 0;"><b>Neue Inserate:</b></td><td>{run.get('total_new', 0)}</td></tr>
        <tr><td style="padding:2px 12px 2px 0;"><b>Aktualisiert:</b></td><td>{run.get('total_updated', 0)}</td></tr>
        <tr><td style="padding:2px 12px 2px 0;"><b>Duplikate abgefangen:</b></td><td>{run.get('total_duplicates_skipped', 0)}</td></tr>
        <tr><td style="padding:2px 12px 2px 0;"><b>Laufzeit:</b></td><td>{format_duration(elapsed_s)}</td></tr>
        <tr><td style="padding:2px 12px 2px 0;"><b>Gesch\u00e4tzte Restzeit:</b></td><td>{eta_str}</td></tr>
        <tr><td style="padding:2px 12px 2px 0;"><b>Letzte Aktualisierung:</b></td><td>{last_update.strftime('%Y-%m-%d %H:%M:%S UTC') if last_update else '\u2014'}</td></tr>
      </table>
    """
    if run.get("error"):
        html += f'<div style="color:#B71C1C; margin-top:8px;"><b>Fehler:</b> {run["error"]}</div>'
    html += "</div>"
    return HTML(html)


latest = runs.find_one(sort=[("started_at", -1)])
display(render_run(latest))

## Live-Refresh

Die folgende Zelle pollt den aktuellsten Run alle 5 Sekunden und rendert ihn neu. Die Schleife endet automatisch, sobald der Run den Status `completed` oder `failed` erreicht. Zum manuellen Abbrechen: Stop-Button in Jupyter (Interrupt Kernel).

In [ ]:
REFRESH_INTERVAL = 5

try:
    while True:
        run = runs.find_one(sort=[("started_at", -1)])
        clear_output(wait=True)
        display(render_run(run))
        if not run or run.get("status") in ("completed", "failed"):
            break
        time.sleep(REFRESH_INTERVAL)
except KeyboardInterrupt:
    print("Live-Refresh abgebrochen.")

## Run-Historie

Die letzten 20 Scraper-L\u00e4ufe:

In [ ]:
rows = []
for r in runs.find(sort=[("started_at", -1)]).limit(20):
    start = r.get("started_at")
    end = r.get("finished_at") or r.get("last_update")
    duration = format_duration((end - start).total_seconds()) if start and end else "\u2014"
    rows.append({
        "started_at": start,
        "status": r.get("status"),
        "pages": f"{r.get('pages_completed', 0)}/{r.get('max_pages', '?')}",
        "new": r.get("total_new", 0),
        "updated": r.get("total_updated", 0),
        "dupes_skipped": r.get("total_duplicates_skipped", 0),
        "duration": duration,
        "error": (r.get("error") or "")[:60],
    })

pd.DataFrame(rows)